## Analysis of STFT vs PSD methods in Classification of AD, FTD, and Healthy Patients

#### Setup and Helper Functions

In [76]:
# Import all necessary libraries and packages for the analysis
import mne
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.signal import stft
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import f1_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

In [77]:
# Setup paths
repo_root = Path.cwd()
data_path = repo_root / "ds004504-download"
eeg_files = sorted(list(data_path.glob("sub-*/eeg/*.set")))
print(f"Found {len(eeg_files)} EEG files.")

Found 88 EEG files.


In [78]:
# Label mapping
participants_file = data_path / "participants.tsv"
if not participants_file.exists():
    raise FileNotFoundError(f"participants.tsv not found at {participants_file}")

participants = pd.read_csv(participants_file, sep='\t')
participants['participant_id'] = participants['participant_id'].str.strip()
print(f"Loaded participants.tsv — {len(participants)} subjects")
print(participants['Group'].value_counts().to_string())

GROUP_MAP = {'A': 1, 'F': 2, 'C': 0}
LABEL_MAP = {0: 'CN', 1: 'AD', 2: 'FTD'}

def get_label(file_path: Path) -> int:
    sub_id = file_path.parts[-3]  # e.g. 'sub-063'
    row = participants[participants['participant_id'] == sub_id]
    if row.empty:
        print(f"  WARNING: {sub_id} not found in participants.tsv — defaulting to CN")
        return 0
    group_str = str(row['Group'].values[0]).strip()
    if group_str not in GROUP_MAP:
        print(f"  WARNING: unknown group '{group_str}' for {sub_id} — defaulting to CN")
        return 0
    return GROUP_MAP[group_str]

Loaded participants.tsv — 88 subjects
Group
A    36
C    29
F    23


### Feature Extraction and Preprocessing

Delta, theta, alpha, beta, and gamma are the five EEG frequency bands used in the paper. When analyzing EEG for AD patients, there is usually an increase in delta/theta power and a decrease in alpha power compared to healthy controls. Additionally, FTD patients usually show increased frontal theta. The classifier will learn to distinguish the changes in the frequency band power.

In [63]:
# Frequency Bands
BANDS = {
    'delta': (0.5,  4),
    'theta': (4,    8),
    'alpha': (8,   13),
    'beta':  (13,  25),
    'gamma': (25,  45),
}

#### Compute Power Spectral Density using Welch's method

In [79]:
# Feature Extraction
def extract_relative_psd(epochs):
    psd_obj = epochs.compute_psd(method='welch', fmin=0.5, fmax=45.0, verbose=False)
    psds, freqs = psd_obj.get_data(return_freqs=True)
    avg_psd     = psds.mean(axis=1)                       
    total_power = avg_psd.sum(axis=1, keepdims=True)      

    feats = []
    for lo, hi in BANDS.values():
        idx = (freqs >= lo) & (freqs <= hi)
        feats.append(avg_psd[:, idx].sum(axis=1, keepdims=True) / total_power)
    return np.hstack(feats)   # (n_epochs, 5)

#### STFT is our experimental method being compared against PSD

In [80]:
def extract_relative_stft(epochs):
    data = epochs.get_data()           
    fs   = int(epochs.info['sfreq'])

    f, _, Zxx   = stft(data, fs=fs, nperseg=256)
    # Average over time frames then over channels
    avg_power   = (np.abs(Zxx) ** 2).mean(axis=-1).mean(axis=1)  

    # Restrict to 0.5–45 Hz
    freq_mask   = (f >= 0.5) & (f <= 45.0)
    avg_power   = avg_power[:, freq_mask]
    f_masked    = f[freq_mask]
    total_power = avg_power.sum(axis=1, keepdims=True)

    feats = []
    for lo, hi in BANDS.values():
        idx = (f_masked >= lo) & (f_masked <= hi)
        feats.append(avg_power[:, idx].sum(axis=1, keepdims=True) / total_power)
    return np.hstack(feats)   # (n_epochs, 5)

#### Preprocessing Loop (PSD and STFT features are extracted for every epoch)

In [83]:
all_psd, all_stft, all_labels, all_groups = [], [], [], []

for subj_id, fpath in enumerate(eeg_files):
    label = get_label(fpath)
    print(f"[{subj_id+1:02d}/{len(eeg_files)}] {fpath.parts[-3]} | "
          f"label={LABEL_MAP[label]}", end=" ... ")
    try:
        raw = mne.io.read_raw_eeglab(fpath, preload=True, verbose=False)
        raw.filter(0.5, 45.0, method='iir', verbose=False)
        raw.set_eeg_reference('average', verbose=False)

        ica = mne.preprocessing.ICA(n_components=15, random_state=97, method='fastica')
        ica.fit(raw, verbose=False)
        ica.apply(raw, exclude=[0], verbose=False)

        epochs = mne.make_fixed_length_epochs(
            raw, duration=4.0, overlap=2.0, preload=True, verbose=False)

        if len(epochs) == 0:
            print("no epochs, skipping.")
            continue

        all_psd.append(extract_relative_psd(epochs))
        all_stft.append(extract_relative_stft(epochs))
        all_labels.extend([label] * len(epochs))
        all_groups.extend([subj_id] * len(epochs))
        print(f"{len(epochs)} epochs")

    except Exception as e:
        print(f"ERROR — {e}")


[01/88] sub-001 | label=AD ... 298 epochs
[02/88] sub-002 | label=AD ... 395 epochs
[03/88] sub-003 | label=AD ... 152 epochs
[04/88] sub-004 | label=AD ... 352 epochs
[05/88] sub-005 | label=AD ... 401 epochs
[06/88] sub-006 | label=AD ... 317 epochs
[07/88] sub-007 | label=AD ... 383 epochs
[08/88] sub-008 | label=AD ... 398 epochs
[09/88] sub-009 | label=AD ... 305 epochs
[10/88] sub-010 | label=AD ... 644 epochs
[11/88] sub-011 | label=AD ... 384 epochs
[12/88] sub-012 | label=AD ... 448 epochs
[13/88] sub-013 | label=AD ... 419 epochs
[14/88] sub-014 | label=AD ... 471 epochs
[15/88] sub-015 | label=AD ... 454 epochs
[16/88] sub-016 | label=AD ... 491 epochs
[17/88] sub-017 | label=AD ... 422 epochs
[18/88] sub-018 | label=AD ... 422 epochs
[19/88] sub-019 | label=AD ... 459 epochs
[20/88] sub-020 | label=AD ... 433 epochs
[21/88] sub-021 | label=AD ... 460 epochs
[22/88] sub-022 | label=AD ... 410 epochs
[23/88] sub-023 | label=AD ... 430 epochs
[24/88] sub-024 | label=AD ... 382

#### Save data

In [84]:
X_psd  = np.vstack(all_psd)
X_stft = np.vstack(all_stft)
y      = np.array(all_labels)
groups = np.array(all_groups)

np.save("X_psd.npy",   X_psd)
np.save("X_stft.npy",  X_stft)
np.save("y.npy",       y)
np.save("groups.npy",  groups)
print("\nFeatures saved to disk.")

print(f"\nTotal epochs: {len(y)}")
for lbl, name in LABEL_MAP.items():
    print(f"  {name}: {(y == lbl).sum()} epochs | "
          f"{len(np.unique(groups[y == lbl]))} subjects")


Features saved to disk.

Total epochs: 35166
  CN: 12175 epochs | 29 subjects
  AD: 14648 epochs | 36 subjects
  FTD: 8343 epochs | 23 subjects


#### We are using four metrics to evaluate binary classification (Accuracy, Sensitivity, Specificity, and F1 score). This is consistent with the paper. 

In [85]:
# Compute Metrics
def compute_metrics(y_true, y_pred, pos_label):
    y_tb = (np.array(y_true) == pos_label).astype(int)
    y_pb = (np.array(y_pred) == pos_label).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_tb, y_pb, labels=[0, 1]).ravel()
    return dict(
        ACC  = (tp + tn) / (tp + tn + fp + fn),
        SENS = tp / (tp + fn) if (tp + fn) > 0 else 0.0,
        SPEC = tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        F1   = f1_score(y_tb, y_pb, zero_division=0)
    )


#### Random Forest Classifier with Leave-One-Subject-Out (LOSO) validation method

In [86]:
# LOSO benchmark
def run_loso(X, y, groups, label_a, label_b, method_name):
    """LOSO cross-validation for a binary task (label_a vs label_b)."""
    task = f"{LABEL_MAP[label_a]} vs {LABEL_MAP[label_b]}"
    mask = np.isin(y, [label_a, label_b])
    X_s, y_s, g_s = X[mask], y[mask], groups[mask]

    if len(np.unique(y_s)) < 2:
        print(f"  [{method_name}] Skipping {task} — only one class present")
        return None

    logo = LeaveOneGroupOut()
    clf  = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    fold_metrics = []

    for train_ix, test_ix in logo.split(X_s, y_s, groups=g_s):
        clf.fit(X_s[train_ix], y_s[train_ix])
        fold_metrics.append(
            compute_metrics(y_s[test_ix], clf.predict(X_s[test_ix]), label_a))

    avg = {k: np.mean([m[k] for m in fold_metrics]) for k in fold_metrics[0]}
    print(f"\n  [{method_name}] {task} ({len(fold_metrics)} folds)")
    print(f"    ACC={avg['ACC']:.4f}  SENS={avg['SENS']:.4f}  "
          f"SPEC={avg['SPEC']:.4f}  F1={avg['F1']:.4f}")
    return avg



In [87]:
print("\n" + "=" * 55)
print("  RESULTS")
print("=" * 55)

for method_name, X_data in [("PSD", X_psd), ("STFT", X_stft)]:
    print(f"\nMETHOD: {method_name}")
    run_loso(X_data, y, groups, 1, 0, method_name)   # AD  vs CN
    run_loso(X_data, y, groups, 2, 0, method_name)   # FTD vs CN


  RESULTS

METHOD: PSD

  [PSD] AD vs CN (65 folds)
    ACC=0.7055  SENS=0.3969  SPEC=0.3087  F1=0.4308

  [PSD] FTD vs CN (52 folds)
    ACC=0.6629  SENS=0.2335  SPEC=0.4294  F1=0.2827

METHOD: STFT

  [STFT] AD vs CN (65 folds)
    ACC=0.6952  SENS=0.3951  SPEC=0.3000  F1=0.4300

  [STFT] FTD vs CN (52 folds)
    ACC=0.6551  SENS=0.2247  SPEC=0.4304  F1=0.2751
